# CAPM: Beta & Annualized Alpha via Variance-Covariance

Data for all S&P-500 stocks, `SPY` as the market, `^IRX` as the risk-free rate, dividend-adjusted closes.
Missing values are left as NaN; `.cov()` handles them pairwise.

In [ ]:
# Importing all important libraries

import numpy as np
import pandas as pd
import yfinance as yf

In [62]:
print("Fetching all S&P 500 tickers from Wikipedia...")




# Fetch the formal S&P 500 table list
wiki_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
sp500_table = pd.read_html(
    wiki_url, 
    storage_options={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
)[0]
stocks = sp500_table['Symbol'].tolist()



# Clean ticker symbols (yfinance uses '-' instead of '.' for tickers like BRK.B)
stocks = [ticker.replace('.', '-') for ticker in stocks]



Fetching all S&P 500 tickers from Wikipedia...


In [ ]:
px = yf.download(stocks + ['SPY', '^IRX'], period="5y", interval="1mo")['Close']  


#      SPY     :     market proxy (dividend-adjusted, so it is comparable with the stocks)
#     ^IRX    :  13-week T-bill yield, quoted in % per year


[*********************100%***********************]  505 of 505 completed


In [51]:
px.head()

Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS,^IRX
Date,,,,,,,,,,,,,,,,,,,,,
2021-10-01,152.187775,146.110596,96.601578,170.660004,117.397575,39.766575,331.259949,650.359985,159.587814,55.771484,...,86.654297,55.002144,54.104977,122.743103,254.500000,113.614059,133.277817,533.950012,204.631027,0.048
2021-11-01,145.819626,161.464157,97.115456,172.539993,114.555779,38.397285,329.976532,669.849976,165.806107,54.309231,...,78.172218,54.269810,50.894260,114.080154,208.330002,112.147385,111.375259,588.780029,210.158508,0.048
2021-12-01,154.275040,173.449448,114.065178,166.490005,128.191010,42.267441,382.741394,567.059998,162.304382,59.005962,...,82.061043,58.046577,52.042446,112.959251,161.509995,126.772934,118.527641,595.200012,230.971832,0.033
2022-01-01,134.806839,170.724182,116.517052,153.970001,116.504570,44.045620,327.296021,534.299988,151.408356,65.474869,...,82.456680,59.727108,64.604408,98.924103,122.290001,114.274582,114.776993,509.119995,189.402847,0.173
2022-02-01,126.137062,161.493820,125.777794,151.490005,110.252510,44.796829,292.528168,467.679993,148.732605,68.850487,...,83.489197,57.729340,67.436050,84.063942,127.500000,112.429543,118.667564,413.339996,183.582062,0.288


* Risk Free Rate (Monthly)

*  Excess Returns (Monthly)


In [52]:
rf  = px['^IRX'] / 100 / 12                                                                                               # annual % -> monthly rate
ret = px[stocks + ['SPY']].pct_change().sub(rf, axis=0)                                                   # monthly excess returns

In [53]:
ret.head()

Ticker,MMM,AOS,ABT,ABBV,ACN,ADBE,AMD,AES,AFL,A,...,WTW,WDAY,WYNN,XEL,XYL,YUM,ZBRA,ZBH,ZTS,SPY
Date,,,,,,,,,,,,,,,,,,,,,
2021-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-11-01,-0.040650,0.081799,-0.024247,0.005280,-0.003914,0.029928,0.317185,-0.064054,0.014533,-0.041884,...,-0.067895,-0.054354,-0.097924,-0.013355,-0.070618,-0.012949,0.102648,-0.164378,0.026972,-0.008075
2021-12-01,0.044609,0.085994,0.119000,0.174504,0.159877,-0.153480,-0.091396,0.039322,0.078473,0.057958,...,0.055098,-0.003856,0.049719,0.069565,-0.009853,0.130386,0.010876,0.064191,0.099009,0.046220
2022-01-01,-0.065505,-0.106689,-0.091308,0.021351,-0.145008,-0.057916,-0.206190,-0.080829,0.075725,-0.126336,...,-0.015008,-0.073978,0.004677,0.028807,-0.124394,-0.098733,-0.144768,-0.031788,-0.180118,-0.052885
2022-02-01,-0.096211,-0.102831,-0.053904,0.079240,-0.106468,-0.124926,0.079322,-0.043072,-0.021684,-0.064553,...,-0.050078,-0.094940,0.012282,-0.033688,-0.150458,-0.016386,-0.188369,0.033657,-0.030972,-0.029757


## Beta

With $\Sigma$ the covariance matrix of the stocks and the market $m$ (SPY):

$$\beta_i = \frac{\Sigma_{i,m}}{\Sigma_{m,m}} = \frac{\text{Cov}(r_i, r_m)}{\text{Var}(r_m)}$$

In [54]:
C    = ret.cov().round(5)                                                        # covariance matrix (pairwise, skips NaNs)
beta = C.loc[stocks, 'SPY'] / C.loc['SPY', 'SPY']                   # Cov(Ri, Rm) / Var(Rm)

In [55]:
ret["SPY"].var().round(5)        # market variance

np.float64(0.00199)

In [56]:
C.head()

Ticker,MMM,AOS,ABT,ABBV,ACN,ADBE,AMD,AES,AFL,A,...,WTW,WDAY,WYNN,XEL,XYL,YUM,ZBRA,ZBH,ZTS,SPY
Ticker,,,,,,,,,,,,,,,,,,,,,
MMM,0.00666,0.00301,0.00281,0.00133,0.00228,0.00257,0.00116,0.00391,0.00229,0.00444,...,0.00161,0.00126,0.00122,0.00177,0.00313,0.00199,0.00475,0.00243,0.00233,0.00200
AOS,0.00301,0.00654,0.00212,0.00103,0.00148,0.00248,0.00326,0.00379,0.00193,0.00269,...,0.00189,0.00054,0.00340,0.00134,0.00363,0.00304,0.00471,0.00185,0.00320,0.00208
ABT,0.00281,0.00212,0.00432,0.00169,0.00284,0.00280,-0.00264,0.00110,0.00186,0.00220,...,0.00287,0.00241,0.00221,0.00097,0.00193,0.00231,0.00213,0.00277,0.00278,0.00108
ABBV,0.00133,0.00103,0.00169,0.00421,0.00005,-0.00060,-0.00122,0.00192,0.00117,0.00094,...,0.00114,-0.00006,0.00122,0.00081,0.00052,0.00160,0.00001,0.00141,0.00078,0.00044
ACN,0.00228,0.00148,0.00284,0.00005,0.01019,0.00747,0.00185,-0.00004,0.00118,0.00419,...,0.00340,0.00795,0.00120,-0.00030,0.00137,0.00005,0.00440,0.00122,0.00332,0.00216


## Alpha

$$\alpha_i^{\text{monthly}} = \mu_i - \beta_i \, \mu_m, \qquad \alpha_i^{\text{annual}} = 12 \times \alpha_i^{\text{monthly}}$$



In [66]:
a     = ret[stocks].mean() - beta * ret['SPY'].mean()                                     # monthly alpha
alpha = a * 12  *100                                                                                       # annualised  alpha

capm_ =  pd.DataFrame({'beta': beta, 'alpha_ann': alpha }).round(4)

In [58]:
capm_

,beta,alpha_ann
Ticker,,
MMM,1.0050,-2.9865
AOS,1.0452,-12.5704
ABT,0.5427,-8.8673
ABBV,0.2211,17.2461
ACN,1.0854,-19.6205
...,...,...
XYL,0.9799,-12.2866
YUM,0.5477,-2.3812
ZBRA,1.6131,-19.3025


In [63]:
##############################################

In [64]:
capm_.loc["AAPL"]

beta         1.0854
alpha_ann    6.2621
Name: AAPL, dtype: float64

In [59]:
capm_.loc["GOOGL"]

beta         1.2010
alpha_ann    6.8353
Name: GOOGL, dtype: float64